In [1]:
import os
import sys
import json
from dotenv import load_dotenv

load_dotenv()

CODE_PRECURATED = os.environ["GPTNL_CODE_PRECURATED"]

sys.path.append(os.environ["GPTNL_SCRATCH"])

In [2]:
counts_epoch_1_fp = (
    os.path.join(os.environ["GPTNL_DATALOADER"], "count_tokens/counts_final_epoch_1.json")
)
counts_epoch_2_fp = (
    os.path.join(os.environ["GPTNL_DATALOADER"], "count_tokens/counts_final_epoch_2.json")
)

counts_epoch_1 = json.load(open(counts_epoch_1_fp, "r"))
counts_epoch_2 = json.load(open(counts_epoch_2_fp, "r"))

code_counts = {
    "epoch_1": {"train": 8.393435e10, "validate": 1.704255e09},
    "epoch_2": {"train": 2.273092e11, "validate": 4.640329e09},
}

counts_epoch_1["train"]["code"] = {"github": code_counts["epoch_1"]["train"]}
counts_epoch_1["validate"]["code"] = {"github": code_counts["epoch_1"]["validate"]}

counts_epoch_2["train"]["code"] = {"github": code_counts["epoch_2"]["train"]}
counts_epoch_2["validate"]["code"] = {"github": code_counts["epoch_2"]["validate"]}

buckets_epoch_1 = list(counts_epoch_1["train"].keys())
buckets_epoch_2 = list(counts_epoch_2["train"].keys())

print(f"Buckets in epoch 1 {len(buckets_epoch_1)}: ", buckets_epoch_1)
print(f"Buckets in epoch 2 {len(buckets_epoch_2)}: ", buckets_epoch_2)

# Count the total tokens in each epoch (train)
total_tokens_train_epoch_1 = sum(
    sum(counts_epoch_1["train"][bucket].values()) for bucket in buckets_epoch_1
)
total_tokens_valid_epoch_1 = sum(
    sum(counts_epoch_1["validate"][bucket].values()) for bucket in buckets_epoch_1
)
total_tokens_train_epoch_2 = sum(
    sum(counts_epoch_2["train"][bucket].values()) for bucket in buckets_epoch_2
)
total_tokens_valid_epoch_2 = sum(
    sum(counts_epoch_1["validate"][bucket].values()) for bucket in buckets_epoch_1
)

print("\nTotal tokens AVAILABLE:")

print(
    f"Total tokens in epoch 1 (train): {total_tokens_train_epoch_1 / 1e9:.2f} billion"
)
print(
    f"Total tokens in epoch 1 (valid): {total_tokens_valid_epoch_1 / 1e9:.2f} billion"
)
print(
    f"Total tokens in epoch 2+ (train): {total_tokens_train_epoch_2 / 1e9:.2f} billion"
)
print(
    f"Total tokens in epoch 2+ (valid): {total_tokens_valid_epoch_2 / 1e9:.2f} billion"
)

Buckets in epoch 1 6:  ['other_languages', 'english_low_medium_quality', 'english_high_quality', 'dutch_low_medium_quality', 'dutch_high_quality', 'code']
Buckets in epoch 2 8:  ['other_languages', 'new_low_medium_quality', 'new_high_quality', 'english_low_medium_quality', 'english_high_quality', 'dutch_low_medium_quality', 'dutch_high_quality', 'code']

Total tokens AVAILABLE:
Total tokens in epoch 1 (train): 355.77 billion
Total tokens in epoch 1 (valid): 7.24 billion
Total tokens in epoch 2+ (train): 538.22 billion
Total tokens in epoch 2+ (valid): 7.24 billion


In [3]:
import pandas as pd

# Create a comprehensive dataframe with all counts
data = []

# Add epoch 1 data
for split, buckets in counts_epoch_1.items():
    for bucket, datasets in buckets.items():
        # Sum all dataset counts within each bucket
        total_count = sum(datasets.values())
        data.append(
            {
                "epoch": "epoch_1",
                "split": split,
                "bucket": bucket,
                "token_count": total_count,
            }
        )

# Add epoch 2 data
for split, buckets in counts_epoch_2.items():
    for bucket, datasets in buckets.items():
        # Sum all dataset counts within each bucket
        total_count = sum(datasets.values())
        data.append(
            {
                "epoch": "epoch_2+",
                "split": split,
                "bucket": bucket,
                "token_count": total_count,
            }
        )

# Create DataFrame
df_overview = pd.DataFrame(data)

# Create a pivot table for better readability
df_pivot = df_overview.pivot_table(
    index=["epoch", "split"], columns="bucket", values="token_count", fill_value=0
)

print("Token counts overview:")
df_pivot

Token counts overview:


bucket                     code  dutch_high_quality  dutch_low_medium_quality  \
epoch    split                                                                  
epoch_1  train     8.393435e+10        1.458467e+10              6.133430e+09   
         validate  1.704255e+09        2.914704e+08              1.264540e+08   
epoch_2+ train     2.273092e+11        2.088095e+10              6.133753e+09   
         validate  4.640329e+09        4.197863e+08              1.253840e+08   

bucket             english_high_quality  english_low_medium_quality  \
epoch    split                                                        
epoch_1  train             4.853224e+10                1.552211e+11   
         validate          9.864046e+08                3.169007e+09   
epoch_2+ train             4.852654e+10                1.552194e+11   
         validate          9.894689e+08                3.170629e+09   

bucket             new_high_quality  new_low_medium_quality  other_languages  
epoch    split                                                                
epoch_1  train         0.000000e+00            0.000000e+00     4.736504e+10  
         validate      0.000000e+00            0.000000e+00     9.652494e+08  
epoch_2+ train         2.444453e+10            8.332165e+09     4.737129e+10  
         validate      4.981740e+08            1.688127e+08     9.683712e+08

In [4]:
# Create a table with total token counts per epoch across categories
# Summing train + validate for each category

# Define category mapping
category_mapping = {
    "english_high_quality": "English (high quality)",
    "english_low_medium_quality": "English (low quality)",
    "dutch_high_quality": "Dutch (high quality)",
    "dutch_low_medium_quality": "Dutch (low quality)",
    "new_high_quality": "Dutch (high quality)",  # New counts as Dutch
    "new_low_medium_quality": "Dutch (low quality)",  # New counts as Dutch
    "code": "Code",
    "other_languages": "Other languages",
}

# Initialize result dictionary
epoch_category_counts = {}

# Process epoch 1
epoch_1_categories = {}
for bucket in counts_epoch_1["train"].keys():
    category = category_mapping[bucket]
    train_count = sum(counts_epoch_1["train"][bucket].values())
    validate_count = sum(counts_epoch_1["validate"][bucket].values())
    total_count = train_count + validate_count

    if category not in epoch_1_categories:
        epoch_1_categories[category] = 0
    epoch_1_categories[category] += total_count

epoch_category_counts["Epoch 1"] = epoch_1_categories

# Process epoch 2+
epoch_2_categories = {}
for bucket in counts_epoch_2["train"].keys():
    category = category_mapping[bucket]
    train_count = sum(counts_epoch_2["train"][bucket].values())
    validate_count = sum(counts_epoch_2["validate"][bucket].values())
    total_count = train_count + validate_count

    if category not in epoch_2_categories:
        epoch_2_categories[category] = 0
    epoch_2_categories[category] += total_count

epoch_category_counts["Epoch 2+"] = epoch_2_categories

# Create DataFrame (categories as rows, epochs as columns)
df_category_totals = pd.DataFrame(epoch_category_counts)

# Convert to billions
df_category_totals = df_category_totals / 1e9

# Calculate percentages for each epoch
epoch_1_total = df_category_totals["Epoch 1"].sum()
epoch_2_total = df_category_totals["Epoch 2+"].sum()

df_category_totals["Epoch 1 %"] = (
    df_category_totals["Epoch 1"] / epoch_1_total * 100
).round(2)
df_category_totals["Epoch 2+ %"] = (
    df_category_totals["Epoch 2+"] / epoch_2_total * 100
).round(2)

# Reorder columns to interleave counts and percentages
df_category_totals = df_category_totals[
    ["Epoch 1", "Epoch 1 %", "Epoch 2+", "Epoch 2+ %"]
]

# Add Total row at the bottom
df_category_totals.loc["Total"] = [epoch_1_total, 100.0, epoch_2_total, 100.0]

# Round to 2 decimal places
df_category_totals = df_category_totals.round(2)

df_category_totals

,Epoch 1,Epoch 1 %,Epoch 2+,Epoch 2+ %
Other languages,48.33,13.31,48.34,8.80
English (low quality),158.39,43.63,158.39,28.84
English (high quality),49.52,13.64,49.52,9.02
Dutch (low quality),6.26,1.72,14.76,2.69
Dutch (high quality),14.88,4.10,46.24,8.42
Code,85.64,23.59,231.95,42.23
Total,363.01,100.00,549.20,100.00


In [5]:
# Create a copy of the pivot table for formatting
df_formatted = df_pivot.copy()

# Convert to billions and round to 2 decimal places
df_formatted = df_formatted / 1e9
df_formatted = df_formatted.round(2)

# Add total rows for each epoch (train + validate)
epochs = df_formatted.index.get_level_values("epoch").unique()
for epoch in epochs:
    epoch_data = df_formatted.loc[epoch]
    total_row = epoch_data.sum()
    # Add the total row to the dataframe
    df_formatted.loc[(epoch, "total"), :] = total_row

# Rename columns to nice looking names
column_mapping = {
    "code": "Code",
    "dutch_high_quality": "Dutch (high quality)",
    "dutch_low_medium_quality": "Dutch (low/medium quality)",
    "english_high_quality": "English (high quality)",
    "english_low_medium_quality": "English (low/medium quality)",
    "new_high_quality": "New (high quality)",
    "new_low_medium_quality": "New (low/medium quality)",
    "other_languages": "Other languages",
    "total": "Total",
}

df_formatted = df_formatted.rename(columns=column_mapping)

print("Available token counts (in billions):")
df_formatted = df_formatted

df_formatted

Available token counts (in billions):


bucket               Code  Dutch (high quality)  Dutch (low/medium quality)  \
epoch    split                                                                
epoch_1  train      83.93                 14.58                        6.13   
         validate    1.70                  0.29                        0.13   
epoch_2+ train     227.31                 20.88                        6.13   
         validate    4.64                  0.42                        0.13   
epoch_1  total      85.63                 14.87                        6.26   
epoch_2+ total     231.95                 21.30                        6.26   

bucket             English (high quality)  English (low/medium quality)  \
epoch    split                                                            
epoch_1  train                      48.53                        155.22   
         validate                    0.99                          3.17   
epoch_2+ train                      48.53                        155.22   
         validate                    0.99                          3.17   
epoch_1  total                      49.52                        158.39   
epoch_2+ total                      49.52                        158.39   

bucket             New (high quality)  New (low/medium quality)  \
epoch    split                                                    
epoch_1  train                   0.00                      0.00   
         validate                0.00                      0.00   
epoch_2+ train                  24.44                      8.33   
         validate                0.50                      0.17   
epoch_1  total                   0.00                      0.00   
epoch_2+ total                  24.94                      8.50   

bucket             Other languages  
epoch    split                      
epoch_1  train               47.37  
         validate             0.97  
epoch_2+ train               47.37  
         validate             0.97  
epoch_1  total               48.34  
epoch_2+ total               48.34

In [6]:
# Create a simplified comparison table with only total counts for each epoch
# First, get the total rows from df_formatted
epoch_1_totals = df_formatted.loc[("epoch_1", "total")]
epoch_2_totals = df_formatted.loc[("epoch_2+", "total")]

# Create comparison dataframe
df_comparison = pd.DataFrame(
    {"Epoch 1 Total": epoch_1_totals, "Epoch 2+ Total": epoch_2_totals}
)

# Calculate percentages for each epoch
epoch_1_total = df_comparison["Epoch 1 Total"].sum()
epoch_2_total = df_comparison["Epoch 2+ Total"].sum()

df_comparison["Epoch 1 %"] = (
    df_comparison["Epoch 1 Total"] / epoch_1_total * 100
).round(2)
df_comparison["Epoch 2+ %"] = (
    df_comparison["Epoch 2+ Total"] / epoch_2_total * 100
).round(2)

# Export to markdown
comparison_markdown = df_comparison.to_markdown()

# Save to file
with open("token_counts.md", "w") as f:
    f.write("# Token Count Comparison: Epoch 1 vs Epoch 2\n\n")
    f.write("Token counts in billions for each data bucket with percentages:\n\n")
    f.write(comparison_markdown)

print("Epoch comparison table with percentages:")
display(df_comparison)
print("\nMarkdown saved to 'token_counts.md'")

Epoch comparison table with percentages:


,Epoch 1 Total,Epoch 2+ Total,Epoch 1 %,Epoch 2+ %
bucket,,,,
Code,85.63,231.95,23.59,42.23
Dutch (high quality),14.87,21.30,4.10,3.88
Dutch (low/medium quality),6.26,6.26,1.72,1.14
English (high quality),49.52,49.52,13.64,9.02
English (low/medium quality),158.39,158.39,43.63,28.84
New (high quality),0.00,24.94,0.00,4.54
New (low/medium quality),0.00,8.50,0.00,1.55
Other languages,48.34,48.34,13.32,8.80



Markdown saved to 'token_counts.md'


# Upsampled sets

In [9]:
buckets_to_lang = {
    "dutch_high_quality": "nl",
    "dutch_low_medium_quality": "nl",
    "english_high_quality": "en",
    "english_low_medium_quality": "en",
    "code": "code",
    "new_high_quality": "nl_new",
    "new_low_medium_quality": "nl_new",
    "other_languages": "other",
}

mult_epoch_1 = {
    "en": 1.3731367800001029,
    "nl": 6.752093888647279,
    "code": 1.0,
    "other": 1.181382596911082,
}
mult_epoch_2 = {
    "en": 1.320775332478761,
    "nl": 1.0,
    "nl_new": 2.73755553476948,
    "code": 0.35517234891407257,
    "other": 1.1361466916188048,
}
mult_epoch_3 = {
    "en": 1.320775332478761,
    "nl": 1.0,
    "nl_new": 2.73755553476948,
    "code": 0.35517234891407257,
    "other": 1.1361466916188048,
}
mult_annealing = {
    "en": 1.3067168401561629,
    "nl": 1.2242529258074888,
    "nl_new": 1.2242529258074888,
    "code": 0.13949528286754942,
    "other": 0.16733471574887998,
}

all_upsampled_counts = {}
all_multipliers = {}

for epoch, multi in zip(
    ["epoch_1", "epoch_2", "annealing"], [mult_epoch_1, mult_epoch_2, mult_annealing]
):
    if epoch == "epoch_1":
        counts = counts_epoch_1["train"]
    else:
        counts = counts_epoch_2["train"]

    upsampled_counts_per_bucket = {}
    multipliers_per_bucket = {}

    for bucket, dataset_counts in counts.items():
        total_count = sum(dataset_counts.values())
        lang = buckets_to_lang[bucket]

        # For annealing, only sample high quality English and Dutch (+ code and other at specified rates)
        if epoch == "annealing":
            if bucket in [
                "english_low_medium_quality",
                "dutch_low_medium_quality",
                "new_low_medium_quality",
            ]:
                multiplier = 0.0
            else:
                multiplier = multi[lang]
        else:
            multiplier = multi[lang]

        upsampled_count = total_count * multiplier
        upsampled_counts_per_bucket[bucket] = upsampled_count
        multipliers_per_bucket[bucket] = multiplier

    all_upsampled_counts[epoch] = upsampled_counts_per_bucket
    all_multipliers[epoch] = multipliers_per_bucket

df_upsampled = pd.DataFrame(all_upsampled_counts)
df_upsampled = df_upsampled / 1e9
df_upsampled["language"] = [
    buckets_to_lang[bucket].replace("nl_new", "nl") for bucket in df_upsampled.index
]

# Create multipliers dataframe
df_multipliers = pd.DataFrame(all_multipliers)
df_multipliers["language"] = [
    buckets_to_lang[bucket].replace("nl_new", "nl") for bucket in df_multipliers.index
]

# Calculate percentages for each epoch
df_percentages = df_upsampled.copy()
for epoch in ["epoch_1", "epoch_2", "annealing"]:
    total = df_percentages[epoch].sum()
    df_percentages[epoch] = (df_percentages[epoch] / total * 100).round(2)

# print("Upsampled token counts (in billions):")
# display(df_upsampled.round(2))

# print("\nUpsampling multipliers per bucket:")
# display(df_multipliers.round(2))

# print("\nPercentage of total upsampled tokens per bucket:")
# display(df_percentages.round(2))

# Concat all together
df_final_upsampled = pd.concat(
    [
        df_upsampled,
        df_multipliers.add_suffix(" Multiplier"),
        df_percentages.add_suffix(" %"),
    ],
    axis=1,
)

# Rename columns
column_rename = {
    "epoch_1": "Epoch 1 Tokens (B)",
    "epoch_1 Multiplier": "Epoch 1 Upsamp. Rate",
    "epoch_1 %": "Epoch 1 %",
    "epoch_2": "Epoch 2 & 3 Tokens (B)",
    "epoch_2 Multiplier": "Epoch 2 & 3 Upsamp. Rate",
    "epoch_2 %": "Epoch 2 & 3 %",
    # 'epoch_3': 'Epoch 3 Tokens (B)',
    # 'epoch_3 Multiplier': 'Epoch 3 Upsamp. Rate',
    # 'epoch_3 %': 'Epoch 3 %',
    "annealing": "Annealing Tokens (B)",
    "annealing Multiplier": "Annealing Upsamp. Rate",
    "annealing %": "Annealing %",
    "language": "Language",
}

df_final_upsampled = df_final_upsampled.rename(columns=column_rename)

# Reorder columns
column_order = [
    "Language",
    "Epoch 1 Tokens (B)",
    "Epoch 1 Upsamp. Rate",
    "Epoch 1 %",
    "Epoch 2 & 3 Tokens (B)",
    "Epoch 2 & 3 Upsamp. Rate",
    "Epoch 2 & 3 %",
    # 'Epoch 3 Tokens (B)', 'Epoch 3 Upsamp. Rate', 'Epoch 3 %',
    "Annealing Tokens (B)",
    "Annealing Upsamp. Rate",
    "Annealing %",
]

df_final_upsampled = df_final_upsampled[column_order]

# Rename the index (bucket names)
index_rename = {
    "code": "Code",
    "dutch_high_quality": "Dutch high quality",
    "dutch_low_medium_quality": "Dutch low quality",
    "english_high_quality": "English high quality",
    "english_low_medium_quality": "English low quality",
    "new_high_quality": "New Dutch high quality",
    "new_low_medium_quality": "New Dutch low quality",
    "other_languages": "Other languages",
}

df_final_upsampled = df_final_upsampled.rename(index=index_rename)

# Add subtotal rows for language groups
lang_groups = df_final_upsampled.groupby("Language").sum(numeric_only=True)

# Create final dataframe with subtotals
df_with_subtotals = pd.DataFrame()
for lang in ["code", "en", "nl", "other"]:
    # Add rows for this language
    lang_rows = df_final_upsampled[df_final_upsampled["Language"] == lang]
    df_with_subtotals = pd.concat([df_with_subtotals, lang_rows])

    # Add subtotal row
    if len(lang_rows) > 0:
        subtotal_row = lang_groups.loc[lang].copy()
        subtotal_row["Language"] = f"{lang.upper()} Subtotal"
        df_with_subtotals.loc[f"{lang}_subtotal"] = subtotal_row

# Add grand total
total_row = df_final_upsampled.sum(numeric_only=True)
total_row["Language"] = "TOTAL"
# Set percentages to 100 for total row
for epoch in ["Epoch 1", "Epoch 2 & 3", "Annealing"]:
    total_row[f"{epoch} %"] = 100.0
df_with_subtotals.loc["grand_total"] = total_row

print(df_with_subtotals.round(2).to_markdown())
df_with_subtotals.round(2)

|                        | Language       |   Epoch 1 Tokens (B) |   Epoch 1 Upsamp. Rate |   Epoch 1 % |   Epoch 2 & 3 Tokens (B) |   Epoch 2 & 3 Upsamp. Rate |   Epoch 2 & 3 % |   Annealing Tokens (B) |   Annealing Upsamp. Rate |   Annealing % |
|:-----------------------|:---------------|---------------------:|-----------------------:|------------:|-------------------------:|---------------------------:|----------------:|-----------------------:|-------------------------:|--------------:|
| Code                   | code           |                83.93 |                   1    |       15    |                    80.73 |                       0.36 |           15.51 |                  31.71 |                     0.14 |         20    |
| code_subtotal          | CODE Subtotal  |                83.93 |                   1    |       15    |                    80.73 |                       0.36 |           15.51 |                  31.71 |                     0.14 |         20    |
| Englis

,Language,Epoch 1 Tokens (B),Epoch 1 Upsamp. Rate,Epoch 1 %,Epoch 2 & 3 Tokens (B),Epoch 2 & 3 Upsamp. Rate,Epoch 2 & 3 %,Annealing Tokens (B),Annealing Upsamp. Rate,Annealing %
Code,code,83.93,1.00,15.00,80.73,0.36,15.51,31.71,0.14,20.00
code_subtotal,CODE Subtotal,83.93,1.00,15.00,80.73,0.36,15.51,31.71,0.14,20.00
English low quality,en,213.14,1.37,38.09,205.01,1.32,39.39,0.00,0.00,0.00
English high quality,en,66.64,1.37,11.91,64.09,1.32,12.32,63.41,1.31,40.00
en_subtotal,EN Subtotal,279.78,2.75,50.00,269.10,2.64,51.71,63.41,1.31,40.00
Dutch low quality,nl,41.41,6.75,7.40,6.13,1.00,1.18,0.00,0.00,0.00
Dutch high quality,nl,98.48,6.75,17.60,20.88,1.00,4.01,25.56,1.22,16.12
New Dutch low quality,nl,NaN,NaN,NaN,22.81,2.74,4.38,0.00,0.00,0.00
New Dutch high quality,nl,NaN,NaN,NaN,66.92,2.74,12.86,29.93,1.22,18.88
nl_subtotal,NL Subtotal,139.89,13.50,25.00,116.74,7.48,22.43,55.49,2.45,35.00


In [10]:
print("Total training tokens after upsampling (in billions), seen throughout training:")
total_dist_per_epoch = df_upsampled.groupby("language").sum()
# Add total row for each epoch
total_dist_per_epoch.loc["total"] = total_dist_per_epoch.sum()

# Create cumulative total across epochs (left to right)
cumulative_data = total_dist_per_epoch.drop("total").cumsum(axis=1)
total_dist_per_epoch.loc["cumulative_total"] = cumulative_data.sum()

total_dist_per_epoch.round(2)

Total training tokens after upsampling (in billions), seen throughout training:


,epoch_1,epoch_2,annealing
language,,,
code,83.93,80.73,31.71
en,279.78,269.10,63.41
nl,139.89,116.74,55.49
other,55.96,53.82,7.93
total,559.56,520.40,158.54
cumulative_total,559.56,1079.96,1238.50
